In [ ]:

import sys
sys.path.insert(1, '../')
import src.data.utils as ut
from src.data.utils import IDX2CAT
from src.model.train import PanImgDataset
from torch.utils.data import DataLoader
from PIL import Image, ImageDraw
import os
import torch
from torchvision import transforms
import pandas as pd
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights,deeplabv3_resnet50,DeepLabV3_ResNet50_Weights
from torchmetrics import  Accuracy, F1Score
from torchmetrics.segmentation import MeanIoU
from tqdm import tqdm
import random
import src.model.train as tr
from src.model import models
import mlflow
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from io import BytesIO
import mlflow
import tempfile

In [ ]:
class PanInference:
    def __init__(self,model_name,state_dict_uri,root_dir,num_classes=33):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.root_dir = root_dir
        self.num_classes = num_classes
        self.model = models.PanSegModel(model_name,self.num_classes)
        self.model.load_state_dict(mlflow.pytorch.load_state_dict(state_dict_uri))
        self.model = self.model.to(self.device)
        self.model.eval()
        self.transform_img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((256, 256)),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
        self.transform_mask = transforms.Compose([
            transforms.PILToTensor(),
            transforms.Resize((256, 256))
            
        ])
        self.pred_tensor = None
        self.true_tensor = None
        self.iou = MeanIoU(num_classes=self.num_classes,per_class=True)
        self.accuracy = Accuracy(task="multiclass", num_classes=self.num_classes,average=None)
        self.f1 = F1Score(task="multiclass", num_classes=self.num_classes,average=None)
        self.metrics_df = None
        self.metrics=None
    
    def get_pred_true_img_mask(self,img_rel_path):
        img_path = os.path.join(self.root_dir,img_rel_path)
        img = Image.open(img_path).convert('RGB')
        trans_img = self.transform_img(img).unsqueeze(0)
        with torch.no_grad():
            out = self.model(trans_img.to(self.device))
        pred_mask = out['out'].argmax(1).detach().cpu()
        pred_mask = transforms.functional.resize(pred_mask,(img.height,img.width))
        
        mask_path = img_path.replace("raw_imgs","mask")+".png"
        true_mask = Image.open(mask_path)
        true_mask_tensor = transforms.functional.pil_to_tensor(true_mask)
        img_tensor = transforms.functional.pil_to_tensor(img)
        return ut.get_masked_img(img_tensor.clone(),pred_mask),ut.get_masked_img(img_tensor.clone(),true_mask_tensor)
    
    def calc_predictions(self,csv_path,batch_size=16):
        test_sets =["pan_set_1","pan_set_2","pan_set_3","pan_set_4"]
        data = pd.read_csv(csv_path).query("set_name in @test_sets")
        val_dataset = PanImgDataset(data,self.root_dir)
        dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


        pred_list =[]
        true_list =[]
        for batch in tqdm(dataloader):
            imgs = batch['img']
            masks = batch['mask']

            with torch.no_grad():
                pred = self.model(imgs.to(self.device))['out']

            self.iou(pred.detach().cpu().argmax(1),masks)
            self.accuracy(pred.detach().cpu().argmax(1),masks)
            self.f1(pred.detach().cpu().argmax(1),masks)
            
        self.metrics_df = pd.DataFrame({
            "iou":pi.iou.compute(),
            "accuracy":pi.accuracy.compute(),
            "f1": pi.f1.compute()
        },index=[IDX2CAT[i] for i in range(self.num_classes)])
        
        self.metrics = {
        "test_iou_bg":self.metrics_df["iou"].mean(),
        "test_accuracy_bg":self.metrics_df["accuracy"].mean(),
        "test_f1_bg":self.metrics_df["f1"].mean(),
        "test_iou_no_bg":self.metrics_df.drop("BACKGROUND")["iou"].mean(),
        "test_accuracy_no_bg":self.metrics_df.drop("BACKGROUND")["accuracy"].mean(),
        "test_f1_no_bg":self.metrics_df["f1"].drop("BACKGROUND").mean(),
    }
    
    def plot_scores(self,scores,categories,title_name='IoU Scores by Category',y_name='IoU Score',x_name='Categories'):
        # Check if the lengths of categories and iou_scores match
        if len(categories) != len(scores):
            raise ValueError("The length of categories and IoU scores must be the same.")

        # Create the bar chart
        plt.figure(figsize=(12, 6))
        plt.bar(categories, scores, color='skyblue')

        # Add titles and labels
        plt.title(title_name)
        plt.xlabel(x_name)
        plt.ylabel(y_name)

        # Display the exact score on each bar
        for i, score in enumerate(scores):
            plt.text(i, score + 0.01, f'{score:.2f}', ha='center', va='bottom')

        # Set y-axis limit (optional)
        plt.ylim(0, 1)

        # Adjust layout
        plt.tight_layout()

        # Save the plot to a BytesIO object
        img_buffer = BytesIO()
        plt.savefig(img_buffer, format='png')
        img_buffer.seek(0)  # Move to the beginning of the buffer
        plt.close()  # Close the plot to free memory

        # Convert BytesIO buffer to PIL Image
        return Image.open(img_buffer)
    
    def get_iou_bar_plot(self):
        no_bg_df = self.metrics_df.drop("BACKGROUND")
        return plot_scores(no_bg_df["iou"].tolist(),no_bg_df.index.tolist())

       
    
        
         

In [ ]:
# state_dict_path="s3://<YOUR_BUCKET>/artifacts/148418839595485829/5518c04033454db9974b29c7caffd9c1/artifacts/models/model_epoch_30"
# state_dict_path="s3://<YOUR_BUCKET>/artifacts/148418839595485829/5d1746b06c7744a49659cb1dc98145c8/artifacts/models/model_epoch_32"
# state_dict_path="s3://<YOUR_BUCKET>/artifacts/148418839595485829/c745f82b44e745f997d85f1be03d3480/artifacts/models/model_epoch_30"
# state_dict_path ="s3://<YOUR_BUCKET>/artifacts/148418839595485829/5518c04033454db9974b29c7caffd9c1/artifacts/models/model_epoch_52"
# state_dict_path ="s3://<YOUR_BUCKET>/artifacts/148418839595485829/5d1746b06c7744a49659cb1dc98145c8/artifacts/models/model_epoch_56"
state_dict_path ="s3://<YOUR_BUCKET>/artifacts/148418839595485829/ea363c06f8054d588a1bcb13367742fb/artifacts/models/model_epoch_44"
root_dir = "/home/ubuntu/Documents/PanaromicSegmentation"
model_name = "deeplabv3_resnet50"
num_classes=33
pi = PanInference(model_name,state_dict_path,root_dir,num_classes)

In [ ]:
csv_path = "/home/ubuntu/Documents/PanaromicSegmentation/data/metadata/test_data.csv"
pi.calc_predictions(csv_path,16)

In [ ]:
pi.metrics_df

In [ ]:
pi.metrics

In [ ]:
pi.get_iou_bar_plot()

## Plot Image

In [ ]:
img_path = "data/raw_imgs/pan_set_3/img_32.jpg"
pred_img,true_img = pi.get_pred_true_img_mask(img_path)
pred_img

In [ ]:
true_img

In [ ]:
img_path = "data/raw_imgs/pan_set_3/img_54.jpg"
pred_img,true_img = pi.get_pred_true_img_mask(img_path)
pred_img

In [ ]:
true_img

In [ ]:
img_path = "data/raw_imgs/pan_set_3/img_78.jpg"
pred_img,true_img = pi.get_pred_true_img_mask(img_path)
pred_img

In [ ]:
true_img

In [ ]:
iou =[0.39,0.47,0.50,0.44,0.43,0.38,0.44,0.49,0.50,0.42,0.38,0.46,0.45,0.22,0.16,0.38,0.42,0.52,0.50,0.38,0.42,0.34,0.39,0.42,0.42,0.28,0.25,0.28,0.31,0.46,0.47,0.43]

In [ ]:
len(iou)

In [ ]:
np.mean(iou)

In [ ]:
acc = [0.56,0.75,0.79,0.74,0.77,0.69,0.68,0.77,0.8,0.68,0.69,0.72,0.79,0.83,0.83,0.70,0.84,0.88,0.88,0.74,0.72,0.77,0.68,0.63,0.59,0.47,0.57,0.58,0.75,0.83,0.87,0.84]

In [ ]:
len(acc)

In [ ]:
np.mean(acc)

In [ ]:
f1 = [0.56,0.64,0.67,0.61,0.60,0.55,0.61,0.66,0.67,0.59,0.55,0.63,0.63,0.36,0.28,0.55,0.60,0.68,0.67,0.55,0.59,0.51,0.56,0.59,0.59,0.43,0.40,0.44,0.48,0.63,0.64,0.6]

In [ ]:
len(f1)


In [ ]:
np.mean(f1)

In [ ]:
txt = '''UR8
UR7
UR6
UR5
UR4
UR3
UR2
UR1
UL1
UL2
UL3
UL4
UL5
UL6
UL7
UL8
LL8
LL7
LL6
LL5
LL4
LL3
LL2
LL1
LR1
LR2
LR3
LR4
LR5
LR6
LR7
LR8'''

In [ ]:
classes = txt.split("\n")

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame({
    "IOU score":iou,
    "Accuracy": acc,
    "F1": f1
},index=classes)

In [ ]:
import numpy as np
for i in range(1,len(accs)+1):
    print(np.mean(accs[0:i]))

## MLFlow

In [ ]:
def resize_image_proportionately(image, scale_factor):
    """
    Resizes a PIL image proportionately by a given scale factor.

    Parameters:
    - image (PIL.Image.Image): The PIL image to resize.
    - scale_factor (float): The factor by which to scale the image. E.g., 0.5 for 50% of original size.

    Returns:
    - resized_image (PIL.Image.Image): The proportionally resized PIL image.
    """
    # Calculate the new size
    original_size = image.size
    new_size = (int(original_size[0] * scale_factor), int(original_size[1] * scale_factor))
    
    # Resize the image
    resized_image = image.resize(new_size, Image.LANCZOS)
    
    return resized_image

def mlflow_log_df(df, fname):
    # Create a temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as temp_file:
        temp_file_path = fname
        
        # Save the DataFrame to the temporary file as CSV
        df.to_csv(temp_file_path, index=False)
        
        # Log the CSV file as an artifact
        mlflow.log_artifact(temp_file_path, artifact_path='csv')
    
    # Remove the temporary file
    os.remove(temp_file_path)


def mlflow_log_img(pil_img,fname):
    # Create a temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix='.png') as temp_file:
        temp_file_path = fname
        
        # Save the PIL image to the temporary file
        pil_img.save(fname, format='PNG')
    
        # Log the image file as an artifact
        mlflow.log_artifact(temp_file_path, artifact_path='imgs')
    
    # Remove the temporary file
    os.remove(temp_file_path)

In [ ]:
tracking_server_uri = "http://localhost:5000"
mlflow.set_tracking_uri(tracking_server_uri)
mlflow.end_run()
run_id = state_dict_path.split("/")[5]
mlflow.start_run(run_id=run_id)
mlflow.log_metrics(pi.metrics)

In [ ]:
mlflow_log_df(pi.metrics_df,"metrics_df.csv")

In [ ]:
mlflow_log_img(pi.get_iou_bar_plot(),"iou_plot.png")

In [ ]:
img_paths = [
    "data/raw_imgs/pan_set_3/img_32.jpg",
    "data/raw_imgs/pan_set_3/img_54.jpg",
    "data/raw_imgs/pan_set_3/img_78.jpg"  
]
for img_path in tqdm(img_paths):
    pred_img,_ = pi.get_pred_true_img_mask(img_path)
    pred_img =resize_image_proportionately(pred_img,1/4)
    img_name = "_".join(img_path.split("/")[-2:]).split(".")[0]+".png"
    mlflow_log_img(pred_img,img_name)

In [ ]:
"_".join(img_paths[0].split("/")[-2:]).split(".")[0]+".png"

In [ ]:
pred_img.size